%pip install gTTS

%pip install playsound==1.2.2

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import face_recognition
from scipy.spatial import distance
import warnings
from gtts import gTTS
from playsound import playsound
import time
import threading
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
warnings.filterwarnings('ignore')

In [3]:
# Configurable thresholds
EYE_AR_THRESH = 0.25       # EAR threshold for closed eyes
MOUTH_AR_THRESH = 0.25      # MAR threshold for yawning
EYE_CLOSED_TIME = 1.5      # Seconds eyes must be continuously closed
MOUTH_OPEN_TIME = 2.5      # Seconds mouth must be continuously open

# Cooldown to prevent repeated TTS
ALERT_COOLDOWN = 5         # seconds

# No of consecutive frames for a state
DROWSINESS_CONSEC_FRAMES = 3
YAWN_CONSEC_FRAMES = 3
HEAD_TURN_CONSEC_FRAMES = 3

# Global variables to track
eye_closed_start = None
mouth_open_start = None
last_alert_time = 0

In [4]:
# calculate eye aspect ratio
def eye_aspect_ratio(eye):
    A = distance.euclidean(eye[1], eye[5])
    B = distance.euclidean(eye[2], eye[4])
    C = distance.euclidean(eye[0], eye[3])
    ear = (A+B) / (2.0 * C)
    return ear

# calculate mount aspect ratio
def mouth_aspect_ratio(mouth):
    A = distance.euclidean(mouth[2], mouth[10])  # roughly upper/lower near left-center
    B = distance.euclidean(mouth[4], mouth[8])   # roughly upper/lower near right-center
    C = distance.euclidean(mouth[0], mouth[6])   # left-right mouth corners (width)
    return (A + B) / (2.0 * C)

In [5]:
def detect_head_turn(landmarks):
    """
    Detect head turning based on facial landmarks asymmetry
    Returns: -1 (left), 0 (center), 1 (right)
    """
    # Check if we have all required landmarks
    if not all(k in landmarks for k in ['left_eye', 'right_eye', 'nose_tip']):
        return 2  # Indeterminate if landmarks are missing
    
    # Calculate eye center points
    left_eye = landmarks['left_eye']
    right_eye = landmarks['right_eye']
    left_eye_center = np.mean(left_eye, axis=0).astype(int)
    right_eye_center = np.mean(right_eye, axis=0).astype(int)
    
    # Get nose tip (usually middle point)
    nose_tip = landmarks['nose_tip'][2]  # Middle point of nose_tip array
    
    # Calculate horizontal distance ratio
    eye_line_center_x = (left_eye_center[0] + right_eye_center[0]) / 2
    nose_offset = nose_tip[0] - eye_line_center_x
    
    # Normalize by inter-eye distance
    eye_distance = right_eye_center[0] - left_eye_center[0]
    normalized_offset = nose_offset / (eye_distance + 1e-5)  # Avoid division by zero
    
    # Determine head turn direction
    if normalized_offset < -0.1:
        head_turned = True
        return 1  # Right turn
    elif normalized_offset > 0.1:
        head_turned = True
        return -1  # Left turn
    else:
        return 0  # Center

In [ ]:
def face_recognition_by_mediapipe(frame):
    model_path = "face_landmarker.task"  # download from the Face Landmarker models page

    BaseOptions = mp.tasks.BaseOptions
    FaceLandmarker = vision.FaceLandmarker
    FaceLandmarkerOptions = vision.FaceLandmarkerOptions
    VisionRunningMode = vision.RunningMode

    options = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.VIDEO,
        output_facial_transformation_matrixes=True,
        output_face_blendshapes=False,
        num_faces=1
    )

    # frame = mp.Image.create_from_file("frame.jpg")
    result = landmarker.detect(frame)

    landmarks = result.face_landmarks[0]                      # 468 (x,y,z)
    T = result.facial_transformation_matrixes[0]              # 4x4; derive yaw/pitch/roll if needed
    
    if landmarks:
        for feature in landmarks.values():
            pts = np.array(feature, np.int32)
            pts = pts.reshape((-1, 1, 2))
            cv2.polylines(frame, [pts], isClosed=True, color=(0, 255, 0), thickness=1)

        # Convert landmarks to a more usable format
        landmark_dict = {
            'left_eye': np.array(landmarks[33:133]),   # Approximate left eye region
            'right_eye': np.array(landmarks[362:462]), # Approximate right eye region
            'mouth': np.array(landmarks[78:308]),      # Approximate mouth region
            'nose_tip': np.array(landmarks[1:6])       # Approximate nose tip region
        }



In [ ]:
# Resize the image before processing to improve performance
def process_image(frame):
    """
    Process one frame:
    - Detect face landmarks
    - Compute Eye Aspect Ratio (EAR)
    - Compute Mouth Aspect Ratio (MAR)
    - Determine if eyes are closed or mouth is open
    """
    if frame is None:
        raise ValueError("Frame is empty or invalid.")

    # ---- Convert to RGB for face_recognition ----
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # ---- Detect face locations ----
    face_locations = face_recognition.face_locations(rgb_frame, model="hog")

    # Flags
    eye_closed = False
    mouth_open = False
    landmarks = None

    if not face_locations:
        return eye_closed, mouth_open

    # draw face boxes in frame
    for (top, right, bottom, left) in face_locations:
        # Draw a rectangle around the face
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
        
        # Optional: Add label above the face box
        label = "Face"
        y = top - 10 if top - 10 > 10 else top + 10
        cv2.putText(frame, label, (left, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)


    for face_location in face_locations:
        try:
            # Get facial landmarks
            landmarks_list = face_recognition.face_landmarks(rgb_frame, [face_location])
            if not landmarks_list:
                continue

            landmarks = landmarks_list[0]

            # drow landmarks as lines
            if landmarks:
                for feature in landmarks.values():
                    pts = np.array(feature, np.int32)
                    pts = pts.reshape((-1, 1, 2))
                    cv2.polylines(frame, [pts], isClosed=True, color=(0, 255, 0), thickness=1)

            #detect head turn
            turn = detect_head_turn(landmarks)
            if turn == -1:
                cv2.putText(frame, "Head Turned Left", (30, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            elif turn == 1:
                cv2.putText(frame, "Head Turned Right", (30, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            elif turn == 0:
                cv2.putText(frame, "Head Centered", (30, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            else:
                cv2.putText(frame, "Head Position Indeterminate", (30, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

            # Ensure required features exist for drowsiness and yawning detection
            if not all(k in landmarks for k in ["left_eye", "right_eye", "top_lip", "bottom_lip"]):
                continue

            # Convert to numpy arrays
            left_eye = np.array(landmarks["left_eye"])
            right_eye = np.array(landmarks["right_eye"])

            # Combine top & bottom lip landmarks (for outer contour)
            mouth_points = np.array(landmarks["top_lip"] + landmarks["bottom_lip"][::-1])

            # # Top lip: first 6 points (outer)
            # top_lip_outer = landmarks["top_lip"][:6]
            # # Bottom lip: first 6 points (outer), reversed for clockwise order
            # bottom_lip_outer = landmarks["bottom_lip"][:6][::-1]
            # # Combine into 12 points for MAR
            # mouth_points = np.array(top_lip_outer + bottom_lip_outer)

            # ---- Compute EAR & MAR ----
            left_ear = eye_aspect_ratio(left_eye)
            right_ear = eye_aspect_ratio(right_eye)
            ear = (left_ear + right_ear) / 2.0
            mar = mouth_aspect_ratio(mouth_points[:12])

            # ---- Determine Drowsiness Indicators ----
            if ear < EYE_AR_THRESH:
                eye_closed = True

            if mar > MOUTH_AR_THRESH:
                mouth_open = True

            # ---- Visualization (optional) ----
            # visualize_landmarks(frame, landmarks)

            # Display EAR & MAR on screen
            cv2.putText(frame, f"EAR: {ear:.3f}", (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
            cv2.putText(frame, f"MAR: {mar:.3f}", (30, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        except Exception as e:
            print(f"[WARN] Error processing landmarks: {e}")
            continue

    return eye_closed, mouth_open

In [ ]:
def estimate_head_pose(image, landmarks):
    """
    Estimate head pose using 3D model points and dlib's predictor
    """
    # 3D model points (standard face model)
    model_points = np.array([
        (0.0, 0.0, 0.0),              # Nose tip
        (0.0, -330.0, -65.0),         # Chin
        (-225.0, 170.0, -135.0),      # Left eye left corner
        (225.0, 170.0, -135.0),       # Right eye right corner
        (-150.0, -150.0, -125.0),     # Left mouth corner
        (150.0, -150.0, -125.0)       # Right mouth corner
    ])
    
    # 2D points from facial landmarks
    image_points = np.array([
        landmarks['nose_tip'][2],     # Nose tip
        landmarks['chin'][8],         # Chin
        landmarks['left_eye'][0],     # Left eye left corner
        landmarks['right_eye'][3],    # Right eye right corner
        landmarks['top_lip'][0],      # Left mouth corner
        landmarks['top_lip'][6]       # Right mouth corner
    ], dtype=np.float32)
    
    # Camera internals
    size = image.shape
    focal_length = size[1]
    center = (size[1]/2, size[0]/2)
    camera_matrix = np.array(
        [[focal_length, 0, center[0]],
         [0, focal_length, center[1]],
         [0, 0, 1]], dtype=np.float32
    )
    
    # Solve for pose
    dist_coeffs = np.zeros((4, 1))
    _, rotation_vector, translation_vector = cv2.solvePnP(
        model_points, image_points, camera_matrix, dist_coeffs)
    
    # Convert rotation vector to usable Euler angles
    rotation_matrix, _ = cv2.Rodrigues(rotation_vector)
    pose_mat = cv2.hconcat((rotation_matrix, translation_vector))
    _, _, _, _, _, _, euler_angles = cv2.decomposeProjectionMatrix(pose_mat)
    
    # Return pitch, yaw, roll in degrees
    return tuple(euler_angles.flatten())

In [7]:
def text_to_speech(text):
    def _play_sound(text_inner):
        try:
            tts = gTTS(text=text_inner, lang='en')
            filename = "alert.mp3"
            tts.save(filename)
            playsound(filename)
            os.remove(filename)
        except Exception as e:
            print(f"[TTS Error] {e}")

    # Run TTS in a separate thread
    t = threading.Thread(target=_play_sound, args=(text,))
    t.daemon = True  # ensures thread exits when main program exits
    # t.start()

In [ ]:
def real_time_detection():
    global eye_closed_start, mouth_open_start, last_alert_time
    cap = cv2.VideoCapture(0)

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            eye_closed, mouth_open = process_image(frame)
            current_time = time.time()

            # ---- Eye Drowsiness Scoring ----
            if eye_closed:
                if eye_closed_start is None:
                    eye_closed_start = current_time  # Start counting
                elif current_time - eye_closed_start >= EYE_CLOSED_TIME:
                    # Trigger alert if cooldown passed
                    if current_time - last_alert_time >= ALERT_COOLDOWN:
                        text_to_speech("Drowsiness detected, please wake up!")
                        last_alert_time = current_time
                    cv2.putText(frame, "DROWSY!", (30, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 3)
            else:
                eye_closed_start = None  # Reset counter

            # ---- Yawning Scoring ----
            if mouth_open:
                if mouth_open_start is None:
                    mouth_open_start = current_time
                elif current_time - mouth_open_start >= MOUTH_OPEN_TIME:
                    if current_time - last_alert_time >= ALERT_COOLDOWN:
                        text_to_speech("Yawning detected, please take a rest.")
                        last_alert_time = current_time
                    cv2.putText(frame, "YAWNING!", (30, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 3)
            else:
                mouth_open_start = None

            # Display EAR/MAR (optional)
            # cv2.putText(frame, f"EAR: {ear:.2f}", (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
            # cv2.putText(frame, f"MAR: {mar:.2f}", (30, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

            cv2.imshow("SafeDriver Monitor", frame)
            if cv2.waitKey(1) & 0xFF == 27:  # ESC to exit
                break

    except Exception as e:
        print(f"[ERROR] {e}")

    finally:
        cap.release()
        cv2.destroyAllWindows()

In [11]:
real_time_detection()